# 25 — Architecture and Design Patterns in Python

Goal: build maintainable systems: clean boundaries, testable design, and idiomatic patterns.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U pydantic
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Composition over inheritance (again)

In Python, “objects with behavior” + duck typing usually beats deep class hierarchies.
Prefer:
- small functions
- small classes with clear responsibilities
- protocols/interfaces for boundaries

## 2.
L2: Dependency injection (make code testable)

Instead of importing global singletons, pass dependencies in.
This makes mocking/testing straightforward.

In [ ]:

from collections.abc import Callable

def compute_report(fetch: Callable[[str], dict], user_id: str) -> str:
    user = fetch(f"/users/{user_id}")
    return f"{user['name']} (active={user['active']})"

# swap real fetch for a fake in tests
fake = lambda path: {"name": "Ada", "active": True}
print(compute_report(fake, "1"))


## 3.
L3: Strategy pattern with callables

“Strategy” is often just passing a function.

In [ ]:

from collections.abc import Callable

def sort_items(items: list[str], key_fn: Callable[[str], object]) -> list[str]:
    return sorted(items, key=key_fn)

words = ["banana", "fig", "apple", "kiwi"]
print(sort_items(words, key_fn=len))
print(sort_items(words, key_fn=lambda s: s[-1]))


## 4.
L4: Factory pattern (usually a function)

Factories are useful when construction logic is non-trivial.

In [ ]:

from dataclasses import dataclass

@dataclass
class Client:
    base_url: str
    timeout: float

def make_client(env: str) -> Client:
    if env == "prod":
        return Client("https://api.example.com", timeout=3.0)
    return Client("https://api.dev.example.com", timeout=10.0)

print(make_client("dev"))


## 5.
L5: Boundary modules (“ports and adapters”)

A robust structure:
- core logic is pure / deterministic
- edges handle I/O: HTTP, DB, filesystem, CLI
- core depends on abstractions (Protocol), edges implement them

This keeps tests fast and reliable.

## 6.
L6: Exercises

1. Refactor a script so parsing/IO is separate from business logic.
2. Use Protocols to define a `Storage` boundary, and implement `InMemoryStorage`.
3. Create tests that swap real storage for fake storage.